# Licență — evaluare 7B baseline pe 2× T4 GPU

Rulează același set de aur prin **Qwen2.5-7B-Instruct-Q4_K_M** via Ollama pe **două GPU-uri T4 în paralel**, pentru comparație cu rulul local 3B pe M1.

**Setup necesar înainte de Run All:**
1. Accelerator: **GPU T4 x2** (în Settings panel, dreapta)
2. Internet: **On**
3. Dataset adăugat: `licenta` (de la `glodcosmin`)

**Optimizări vs versiunea single-GPU:**
- Două instanțe Ollama, una pe fiecare GPU, fiecare cu `OLLAMA_NUM_PARALLEL=2`
- ThreadPoolExecutor cu 4 workers care dispatch round-robin către cele două instanțe
- Timp așteptat: ~8–12 min pentru ~60 de cazuri (vs ~30 min secvențial pe un singur GPU)

## 1. Pregătire mediu

Dezinstalăm `torchvision` și `torchaudio` (pre-instalate de Kaggle, blochează importul `sentence-transformers` pe combinația noastră de torch).

In [ ]:
!pip uninstall -y torchvision torchaudio

In [ ]:
!nvidia-smi

## 2. Instalează Ollama

Instalatorul descarcă un binar `.tar.zst`, deci avem nevoie de `zstd`.

In [ ]:
!sudo apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Pornește 2 instanțe Ollama, câte una pe fiecare GPU

Pinăm fiecare server la un GPU diferit prin `CUDA_VISIBLE_DEVICES` și un port diferit:
- GPU 0 → port **11434**
- GPU 1 → port **11435**

Fiecare instanță poate servi 2 cereri simultan (`OLLAMA_NUM_PARALLEL=2`). Cele două instanțe partajează același cache local de modele (`~/.ollama/models/`), deci `ollama pull` se face o singură dată.

In [ ]:
import subprocess, time, os, urllib.request

# Stop any prior instance
subprocess.run(['pkill', '-f', 'ollama serve'], capture_output=True)
time.sleep(3)

HOSTS = []
for gpu_id, port in [(0, 11434), (1, 11435)]:
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['OLLAMA_HOST'] = f'127.0.0.1:{port}'
    env['OLLAMA_NUM_PARALLEL'] = '2'
    env['OLLAMA_MAX_LOADED_MODELS'] = '1'
    log = open(f'/tmp/ollama_gpu{gpu_id}.log', 'w')
    subprocess.Popen(['ollama', 'serve'], env=env, stdout=log, stderr=log)
    HOSTS.append(f'http://127.0.0.1:{port}')

# Wait for both APIs to come up
for host in HOSTS:
    for i in range(30):
        try:
            urllib.request.urlopen(f'{host}/api/version', timeout=2)
            print(f'{host}: UP after {i+1}s')
            break
        except Exception:
            time.sleep(1)
    else:
        !tail -30 /tmp/ollama_gpu*.log
        raise RuntimeError(f'{host} did not come up')

## 4. Pull modelul Qwen2.5-7B-Q4_K_M (~4.5 GB)

Se descarcă o singură dată în `~/.ollama/models/`. Ambele instanțe îl vor vedea.

In [ ]:
import os
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
!ollama pull qwen2.5:7b-instruct-q4_K_M

## 5. Verifică structura dataset-ului

In [ ]:
!find /kaggle/input -maxdepth 4 -type d 2>/dev/null

## 6. Copiază codul + datele în spațiul writable

`/kaggle/input/` e read-only, iar Chroma + cache-ul HuggingFace au nevoie să scrie.

In [ ]:
import shutil, sys, os
from pathlib import Path

SRC = Path('/kaggle/input/datasets/glodcosmin/licenta')
WORK = Path('/kaggle/working/licenta')

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir()

shutil.copytree(SRC / 'src', WORK / 'src')
(WORK / 'data' / 'eval').mkdir(parents=True)
shutil.copy(SRC / 'data' / 'cezicelegea_dump.json', WORK / 'data' / 'cezicelegea_dump.json')
shutil.copy(SRC / 'data' / 'eval' / 'gold_set.json', WORK / 'data' / 'eval' / 'gold_set.json')

sys.path.insert(0, str(WORK / 'src'))
os.chdir(WORK)
print(f'Working dir: {os.getcwd()}')
!ls -R . | head -40

## 7. Instalează dependențele Python

**Nu** pinăm `torch` / `transformers` — folosim ce e pre-instalat de Kaggle (compatibil cu cuda). Avertismentele dep-resolver despre `timm`, `gradio`, `opentelemetry` etc. sunt benigne (sunt despre alte pachete pre-instalate).

In [ ]:
!pip install -q \
    'chromadb>=1.5' \
    'sentence-transformers<4' \
    'pydantic>=2.13' \
    'ollama>=0.6' \
    'fpdf2'

## 8. Construiește indexul Chroma

Descarcă bge-m3 (~2.3 GB, 30–60s pe Kaggle), apoi calculează embedding-urile pentru cele 149 de chunk-uri (~15s pe T4).

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

from licenta.index import build_index
build_index()

## 9. Warm both Ollama instances + activează rutarea per-thread

Trimite o cerere mică către fiecare instanță pentru a încărca modelul în VRAM. Apoi monkey-patch-uim `ollama.chat` să folosească un host configurat per thread — fără modificări în codul `licenta.generator`.

In [ ]:
import threading
import ollama

MODEL = 'qwen2.5:7b-instruct-q4_K_M'

# Pre-load model on each GPU instance
for host in HOSTS:
    print(f'Warming {host}...', end=' ', flush=True)
    ollama.Client(host=host).chat(
        model=MODEL,
        messages=[{'role': 'user', 'content': 'hi'}],
    )
    print('OK')

# Thread-local routing: each worker thread will set _tls.host before calling ollama.chat
_tls = threading.local()
_orig_chat = ollama.chat

def _routing_chat(*args, **kwargs):
    host = getattr(_tls, 'host', None)
    if host:
        return ollama.Client(host=host).chat(*args, **kwargs)
    return _orig_chat(*args, **kwargs)

ollama.chat = _routing_chat

print(f'\nGata: {len(HOSTS)} GPU × 2 paralel = 4 workers concurenți')

## 10. Sanity-test: o singură întrebare prin pipeline-ul 7B

Verificăm că răspunsul vine în română și conține o citație. Dacă da, putem da drumul la evaluarea completă.

In [ ]:
from licenta.retriever import Retriever
from licenta.generator import generate

retriever = Retriever()
hits = retriever.query('Care este vârsta minimă pentru căsătorie?', k=4)
answer = generate('Care este vârsta minimă pentru căsătorie?', hits, model=MODEL)

print('Răspuns:', answer.text)
print('Surse citate:', answer.schema.cited_source_indices)
print('Valid:', answer.valid)
print('Attempts:', answer.attempts)

## 11. Rulare paralelă a evaluării 7B (4 workers concurenți)

ThreadPoolExecutor dispatchează cele 60 de cazuri round-robin către cele 4 sloturi (2 GPU × 2 paralel). Pe T4 ar trebui să dureze **~8–12 min** total (vs ~30 min serial pe un singur GPU).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict
from pathlib import Path
from tqdm.auto import tqdm
import json, time

from licenta.evaluate import load_gold_set, evaluate_case, _build_report

cases = load_gold_set(Path('data/eval/gold_set.json'))
print(f'Evaluez {len(cases)} cazuri pe {len(HOSTS)} GPU × 2 paralel...')

def worker(idx_case):
    idx, case = idx_case
    _tls.host = HOSTS[idx % len(HOSTS)]
    try:
        return evaluate_case(case, retriever, MODEL, k=4)
    except Exception as e:
        print(f'  ERROR pe {case.id}: {e}')
        return None

start = time.time()
indexed = list(enumerate(cases))
results = []
with ThreadPoolExecutor(max_workers=len(HOSTS) * 2) as ex:
    for r in tqdm(ex.map(worker, indexed), total=len(cases), desc='eval'):
        if r is not None:
            results.append(r)

elapsed = time.time() - start
print(f'\nTotal: {elapsed/60:.1f} min ({elapsed/len(cases):.1f}s per caz)')

# Sort by case_id for deterministic output
results.sort(key=lambda r: r.case_id)

# Write outputs
out_dir = Path('/kaggle/working/eval_7b')
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'results.json').write_text(
    json.dumps([asdict(r) for r in results], ensure_ascii=False, indent=2)
)
(out_dir / 'report.md').write_text(_build_report(results, MODEL, k=4))
print(f'Wrote {out_dir}/results.json și report.md')

## 12. Afișează raportul

In [ ]:
from IPython.display import Markdown, display
with open('/kaggle/working/eval_7b/report.md') as f:
    display(Markdown(f.read()))

## 13. Descărcare rezultate

Fișierele din `/kaggle/working/eval_7b/` apar în panoul **Output** (dreapta sus). Click pe `report.md` și `results.json` ca să le descarci.

Local: pune-le în `data/eval/runs/<timestamp>_qwen2.5_7b-instruct-q4_K_M/` pentru comparație directă cu rulul 3B.